# 10 — Build the analytical panel

Merge trade, demand and refinery output into one annual product panel and derive transparent dependence metrics.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.metrics import add_supply_metrics, add_yoy
from portugal_refining_resilience.validation import assert_unique


In [ ]:
trade = pd.read_csv(PATHS.processed / "fuel_trade_annual.csv")
trade_wide = trade.pivot(index=["year", "product"], columns="flow", values="value_kt").reset_index()
trade_wide.columns.name = None
trade_wide = trade_wide.rename(columns={"imports": "imports_kt", "exports": "exports_kt"})
demand = pd.read_csv(PATHS.processed / "fuel_demand_annual.csv")
output = pd.read_csv(PATHS.processed / "fuel_refinery_output_annual.csv")
panel = trade_wide.merge(demand[["year", "product", "demand_kt"]], on=["year", "product"], how="outer", validate="one_to_one")
panel = panel.merge(output[["year", "product", "refinery_output_kt", "refining_regime"]], on=["year", "product"], how="outer", validate="one_to_one")
panel = add_supply_metrics(panel)
panel = add_yoy(panel, ["imports_kt", "exports_kt", "demand_kt", "refinery_output_kt", "net_imports_kt"])
assert_unique(panel, ["year", "product"])
persist_dataframe(panel, PATHS.processed / "fuel_annual_analytical_panel.csv", key_columns=["year", "product"])
display(panel.tail(10))


In [ ]:
missingness = panel.isna().groupby(panel["product"]).mean().T
missingness.to_csv(PATHS.metrics / "annual_panel_missingness.csv")
display(missingness)
